In [1]:
import os
import time
import random
import hashlib
import logging
from typing import Optional, Dict, Any
import pandas as pd
from bs4 import BeautifulSoup
from curl_cffi import requests

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Configuration ---

# Input/Output Dirs
# Updated to point to the correct subdirectory for input files
INPUT_FOLDER = r"C:\Users\Matt Shaver\OneDrive\Desktop\Football Reruitment Tables\Recruitment Classes"
OUTPUT_FOLDER = r"C:\Users\Matt Shaver\OneDrive\Desktop\Football Reruitment Tables\Scouting Reports"
CACHE_DIR = "247_profile_cache"

# Ensure output directory exists
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

START_YEAR = 2015
END_YEAR = 2015   ##2026

# Rate Limiting
MIN_DELAY = 1.0
MAX_DELAY = 3.0

print(f"Reading recruits from: {INPUT_FOLDER}")
print(f"Saving reports to: {OUTPUT_FOLDER}")

Reading recruits from: C:\Users\Matt Shaver\OneDrive\Desktop\Football Reruitment Tables\Recruitment Classes
Saving reports to: C:\Users\Matt Shaver\OneDrive\Desktop\Football Reruitment Tables\Scouting Reports


In [2]:
class ProfileScraper:
    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir
        self.session = requests.Session()

    def _get_cache_path(self, url: str) -> str:
        url_hash = hashlib.md5(url.encode("utf-8")).hexdigest()
        return os.path.join(self.cache_dir, f"{url_hash}.html")

    def _sleep_randomly(self):
        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    def fetch_page(self, url: str) -> Optional[str]:
        cache_path = self._get_cache_path(url)
        
        # Try Cache
        if os.path.exists(cache_path) and os.path.getsize(cache_path) > 0:
            try:
                with open(cache_path, "r", encoding="utf-8") as f:
                    return f.read()
            except Exception:
                pass

        # Fetch Live
        for attempt in range(3):
            try:
                self._sleep_randomly()
                response = self.session.get(
                    url, 
                    impersonate="chrome", 
                    timeout=30,
                    headers={"User-Agent": "Mozilla/5.0"}
                )
                
                if response.status_code == 200:
                    with open(cache_path, "w", encoding="utf-8") as f:
                        f.write(response.text)
                    return response.text
                elif response.status_code == 404:
                    logger.warning(f"404 Not Found: {url}")
                    return None
                
                time.sleep(2)
            except Exception as e:
                logger.error(f"Fetch error {url}: {e}")
                time.sleep(2)
        
        return None

    def parse_profile(self, html: str, url: str) -> Dict[str, Any]:
        soup = BeautifulSoup(html, "html.parser")
        data = {"url": url, "athletic_background": None, "skills": {}}
        
        # --- Parse H.S. Athletic Background ---
        # Search for headers containing "Athletic Background"
        bg_headers = soup.find_all(lambda tag: tag.name in ["h2", "h3", "h4", "h5", "b", "strong"] and "Athletic Background" in tag.get_text())
        
        for bg_header in bg_headers:
            # Look at siblings coming after the header
            # We want to find the first substantial text block
            sibling_limit = 5
            attempts = 0
            
            for sibling in bg_header.next_siblings:
                attempts += 1
                if attempts > sibling_limit:
                    break
                
                # If we hit another header or section, stop
                if sibling.name in ["h2", "h3", "h4", "h5", "ul", "section"]:
                    break
                    
                text_content = ""
                if isinstance(sibling, str):
                    text_content = sibling.strip()
                elif sibling.name in ["p", "div", "span"]:
                    text_content = sibling.get_text(strip=True)
                
                # Valid text found?
                if text_content and len(text_content) > 3: # Ignore tiny punctuation
                    data["athletic_background"] = text_content
                    break
            
            if data["athletic_background"]:
                break
        
        # Fallback: specific classes if generic header search failed
        if not data["athletic_background"]:
            bg_div = soup.select_one(".background-content")
            if bg_div:
                data["athletic_background"] = bg_div.get_text(strip=True)

        # --- Parse Skills ---
        skills_section = soup.find(lambda tag: tag.name in ["h2", "h3", "h4", "h5"] and "Skills" in tag.get_text())
        if skills_section:
            container = skills_section.find_next("ul")
            if container:
                for li in container.find_all("li"):
                    # Attempt to extract label and score
                    label, score = None, None
                    
                    # Method 1: Split text by whitespace (simplest fallback)
                    full_text = li.get_text(" ", strip=True)
                    parts = full_text.split()
                    
                    if parts and parts[-1].isdigit():
                        score = parts[-1]
                        # Join the rest as label
                        # Aggressively clean the label to prevent csv/excel errors
                        raw_label = " ".join(parts[:-1])
                        
                        # Remove Excel injection characters (=, -, +) from start
                        # Remove colons, dashes from end
                        label = raw_label.lstrip("=-+").rstrip(":- ").strip()
                        
                    if label and score:
                        data["skills"][label] = score

        return data

In [ ]:
# --- Main Execution ---
scraper = ProfileScraper(CACHE_DIR)

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n--- Processing Year: {year} ---")
    
    input_file = os.path.join(INPUT_FOLDER, f"recruits_{year}.csv")
    output_file = os.path.join(OUTPUT_FOLDER, f"scouting_reports_{year}.csv")
    
    if not os.path.exists(input_file):
        logger.warning(f"Input file not found: {input_file}. Skipping year.")
        continue
        
    try:
        df = pd.read_csv(input_file)
        
        # Check for required columns
        if "url" not in df.columns or "player_id" not in df.columns:
            logger.warning(f"Missing 'url' or 'player_id' column in {input_file}. Skipping.")
            continue
            
        # Get unique profiles (player_id, url)
        # Drop rows where URL is missing
        subset = df[["player_id", "url"]].dropna(subset=["url"])
        # Remove duplicates based on URL to avoid rescraping same page
        subset = subset.drop_duplicates(subset=["url"])

        # --- TEST LIMIT ---
        TEST_LIMIT = 50
        if TEST_LIMIT:
            print(f"TEST MODE ENABLED: Limiting to first {TEST_LIMIT} records.")
            subset = subset.head(TEST_LIMIT)
        
        records = subset.to_dict("records")
        print(f"Found {len(records)} profiles to scrape.")
        
        results = []
        
        for i, record in enumerate(records):
            url = record["url"]
            pid = record["player_id"]
            
            if i % 10 == 0:
                print(f"  Scraping {i}/{len(records)}", end="\r")
                
            html = scraper.fetch_page(url)
            if html:
                data = scraper.parse_profile(html, url)
                
                # Format to specific columns:
                # player_id, hs_athletic_background, skill_1, skill_1_rating, ... skill_8, skill_8_rating
                
                flat_data = {
                    "player_id": pid,
                    "hs_athletic_background": data.get("athletic_background")
                }
                
                # Flatten skills into 8 fixed slots
                skills_dict = data.get("skills", {})
                skill_items = list(skills_dict.items()) # List of (Name, Rating) tuples
                
                for idx in range(1, 9):
                    s_key = f"skill_{idx}"
                    r_key = f"skill_{idx}_rating"
                    
                    if idx <= len(skill_items):
                        s_name, s_val = skill_items[idx-1]
                        flat_data[s_key] = s_name
                        flat_data[r_key] = s_val
                    else:
                        flat_data[s_key] = None
                        flat_data[r_key] = None
                    
                results.append(flat_data)
        
        if results:
            # Define exact column order
            cols = ["player_id", "hs_athletic_background"]
            for idx in range(1, 9):
                cols.append(f"skill_{idx}")
                cols.append(f"skill_{idx}_rating")
            
            out_df = pd.DataFrame(results)
            # Ensure all columns exist even if data was empty
            for col in cols:
                if col not in out_df.columns:
                    out_df[col] = None
            
            # Reorder columns matches screenshot
            out_df = out_df[cols]
            
            out_df.to_csv(output_file, index=False, encoding="utf-8-sig")
            print(f"\nSaved {len(out_df)} reports to {output_file}")
        else:
            print(f"\nNo data found for {year}.")
            
    except Exception as e:
        logger.error(f"Error processing {year}: {e}")

_IncompleteInputError: incomplete input (1052043859.py, line 20)